# 11 하이브리드 수요예측

**type별 SBC(rule-base) vs ML 클러스터링** — 제품수(검증 판매량) 반영 **가중 WMAPE** 비교

> 논문: 변동성 큰 **B센터**에서는 SBC가, 저변동 **A센터**에서는 ML이 유리했음.  
> 본 데이터는 type별 변동성 차이가 상대적으로 작아 **결과가 항상 같지 않으며**, 센터·데이터 특성에 따라 우세 scheme이 달라질 수 있음.

### ⓪ 환경 설정

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())

phase1_all = pd.read_parquet(DATA_PROCESSED / 'phase1_all_results.parquet')
phase2 = pd.read_parquet(DATA_PROCESSED / 'phase2_results.parquet')
phase1_best = pd.read_csv(DATA_PROCESSED / 'phase1_best_per_condition.csv')
phase2_best = pd.read_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv')
final_best = pd.read_csv(DATA_PROCESSED / 'final_best_per_condition.csv')


### ① 가중 WMAPE — type별 SBC vs ML

In [ ]:
import numpy as np
import numpy as np
from utils.phase_analysis import (
    validation_weights, build_family_final_results,
    compare_schemes_by_type, summarize_wmape,
)

val_weights = validation_weights(df)
family_final = build_family_final_results(phase1_all, phase2, final_best, val_weights)

type_compare = compare_schemes_by_type(family_final)
display(type_compare)
print('가중 WMAPE 우세:', type_compare['better_scheme_weighted'].value_counts().to_dict())

cv = df[df['yearweek'] <= TRAIN_WEEK_MAX].groupby('type')['sales'].agg(['mean', 'std'])
cv['cv'] = (cv['std'] / cv['mean'].replace(0, np.nan)).round(3)
print('\n=== type별 학습구간 판매 변동계수(CV) ===')
print(cv)


### 해석

- **가중 WMAPE** = Σ(WMAPE_f × 검증판매량_f) / Σ(검증판매량_f)
- 논문과 달리 본 실험에서는 type 간 변동성 격차가 크지 않을 수 있음 → scheme 우세가 센터마다 다르게 나타남
- `final_best_per_condition.csv` → 12장 RIDR 분석 이후 활용